In [5]:
# Track ArUco markers

import cv2
import cv2.aruco as aruco
import numpy as np
import json

# Load the camera calibration parameters from './calibration_data.json'
with open('./calibration_data.json', 'r') as f:
    calibration_data = json.load(f)
    camera_matrix = np.array(calibration_data['camera_matrix'])
    dist_coeffs = np.array(calibration_data['distortion_coefficients'])

# Define the ArUco dictionary and parameters
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
parameters = aruco.DetectorParameters()

# Create detector
detector = aruco.ArucoDetector(aruco_dict, parameters)

inch_to_meter = 0.0254
marker_length = 7 * inch_to_meter  

obj_points = np.array([[-marker_length/2, marker_length/2, 0],
                        [marker_length/2, marker_length/2, 0],
                        [marker_length/2, -marker_length/2, 0],
                        [-marker_length/2, -marker_length/2, 0]], dtype=np.float32)

In [6]:
# Capture a frame from the camera and detect ArUco markers

import matplotlib.pyplot as plt

cap = cv2.VideoCapture(0)  # Use 0 for default camera
if not cap.isOpened():
    print("Cannot open camera")
    exit()

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to capture frame")
            break

        frame = cv2.undistort(frame, camera_matrix, dist_coeffs)

        # Detect markers
        corners, ids, rejected = detector.detectMarkers(frame)

        if ids is not None:
            print(f"Detected markers: {ids.flatten()}")
            
            # Draw detected markers
            aruco.drawDetectedMarkers(frame, corners, ids)
            
            # Estimate pose manually using solvePnP
            for i in range(len(ids)):
                success, rvec, tvec = cv2.solvePnP(obj_points, corners[i], camera_matrix, dist_coeffs)
                if success:
                    print(f"Marker {ids[i][0]}: rvec={rvec.flatten()}, tvec={tvec.flatten()}")
                    cv2.drawFrameAxes(frame, camera_matrix, dist_coeffs, rvec, tvec, 0.1)
                else:
                    print(f"Pose estimation failed for marker {ids[i][0]}")

        cv2.imshow('frame', frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
except KeyboardInterrupt:
    print("Exiting...")
finally:
    cap.release()
    cv2.destroyAllWindows()

Detected markers: [17]
Marker 17: rvec=[-1.01617462  2.16376671  1.94298513], tvec=[-3.30042753 -0.80814305  7.38706048]
Detected markers: [0]
Marker 0: rvec=[-2.02316906  1.95443391 -0.47704391], tvec=[-0.09243877 -0.2813272   1.03011828]
Detected markers: [0]
Marker 0: rvec=[-2.09985052  1.98983546 -0.36935014], tvec=[-0.08562135 -0.24920859  1.05195681]
Detected markers: [0]
Marker 0: rvec=[-2.1056829   1.97567702 -0.1878098 ], tvec=[-0.07910563 -0.2453701   1.05734634]
Detected markers: [0]
Marker 0: rvec=[-2.13425981  2.0192576  -0.03515149], tvec=[-0.07486663 -0.24861782  1.06076335]
Detected markers: [0]
Marker 0: rvec=[-2.19759532  2.09464679  0.07054357], tvec=[-0.07189462 -0.25429025  1.06321753]
Detected markers: [0]
Marker 0: rvec=[-2.23248688  2.12902819  0.0383036 ], tvec=[-0.06998847 -0.25745736  1.06123464]
Detected markers: [0]
Marker 0: rvec=[-2.21662225  2.12237429  0.03891246], tvec=[-0.06817874 -0.2579875   1.06157138]
Detected markers: [0]
Marker 0: rvec=[-2.23633

In [7]:
cap.release()
cv2.destroyAllWindows()